<a href="https://colab.research.google.com/github/suriarasai/BEAD2025/blob/main/05a_Log_Analytics_Using_RDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The Scenario and Data Set

We will process the famous NASA's web server logs from July 1995. Each line in the log file represents a request made to the server and follows the Common Log Format.

Our Goal:

Read the raw log file into an RDD.

Parse each line to extract the HTTP status code (e.g., 200 for OK, 404 for Not Found) and the request URL.

Filter out any malformed log entries that don't parse correctly.

Count the number of times each HTTP status code appears in the entire log.

### Setup: Getting the Data

We need to download the public dataset. We can can run the following command in code using bash operator to fetch the data.

In [30]:
# This command downloads the compressed log file and unzips it.
!wget ftp://ita.ee.lbl.gov/traces/NASA_access_log_Jul95.gz -O NASA_access_log_Jul95.gz
!gunzip NASA_access_log_Jul95.gz

--2025-08-16 04:17:57--  ftp://ita.ee.lbl.gov/traces/NASA_access_log_Jul95.gz
           => ‘NASA_access_log_Jul95.gz’
Resolving ita.ee.lbl.gov (ita.ee.lbl.gov)... 131.243.2.164, 2620:83:8000:102::a4
Connecting to ita.ee.lbl.gov (ita.ee.lbl.gov)|131.243.2.164|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /traces ... done.
==> SIZE NASA_access_log_Jul95.gz ... 20676672
==> PASV ... done.    ==> RETR NASA_access_log_Jul95.gz ... done.
Length: 20676672 (20M) (unauthoritative)

NASA_access_log_Jul 100%[===================>]  19.72M  7.69MB/s    in 2.6s    

2025-08-16 04:18:01 (7.69 MB/s) - ‘NASA_access_log_Jul95.gz’ saved [20676672]

gzip: NASA_access_log_Jul95 already exists; do you wish to overwrite (y or n)? y


This will create a file named NASA_access_log_Jul95 in the workspace.

### PySpark Setup

The first step involves installing pyspark.  The next step is to install findspark library.

*Note: the --ignore-install flag is used to ignore previous installations and use the latest one built alongside the allocated cluster.*


In [31]:
import os

# 1. Install OpenJDK 21 (if not already done in a previous cell)
!apt-get update -qq
!apt-get install -qq openjdk-21-jdk-headless

# 2. Verify where it landed (if needed)
!ls /usr/lib/jvm | grep 21

# 3. Point to JDK 21
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# 4. Install PySpark via pip (make sure this happens AFTER setting JAVA_HOME)
!pip install pyspark --quiet

# 5. Import and start Spark
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
      .master("local[*]")
      .appName("Spark on Java21")
      .getOrCreate()
)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
java-1.21.0-openjdk-amd64
java-21-openjdk-amd64


In PySpark, a Spark Session is created using the SparkSession.builder method. Here's an example:

In [32]:
from pyspark.sql import SparkSession
# import collections
spark = SparkSession.builder.master("local").appName("Log Analytics").getOrCreate()

### Log Processing

Set the Regular Expression Pattern

In [33]:
import re
# A regular expression to parse the Common Log Format.
# Example: 199.72.81.55 - - [01/Jul/1995:00:00:01 -0400] "GET /history/apollo/ HTTP/1.0" 200 6245
LOG_PATTERN = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+)\s*(\S*)" (\d{3}) (\S+)'

Function to parse a log line. Returns a tuple or None if parsing fails.

In [34]:
def parse_log_line(line):
    match = re.search(LOG_PATTERN, line)
    if match:
        # We are interested in the status code (group 8) and the URL (group 6)
        status_code = int(match.group(8))
        url = match.group(6)
        return (status_code, url)
    else:
        return None

Read the text file into an RDD

In [35]:
# Read Log File
log_file_path = "/content/NASA_access_log_Jul95"
# Each line of the file becomes an element in the RDD.
log_rdd = spark.sparkContext.textFile(log_file_path)

Data Munging

In [36]:
parsed_logs_rdd = log_rdd.map(parse_log_line)
# Filter out the lines that failed to parse (returned None)
valid_logs_rdd = parsed_logs_rdd.filter(lambda x: x is not None)

# The RDD is now structured as (status_code, url).
# For our goal, we just need the status code.
# map() -> (200, 1), (404, 1), (200, 1), ...
status_counts_rdd = valid_logs_rdd.map(lambda x: (x[0], 1))

Log Consolidation

In [37]:
# Count the occurrences of each status code
# reduceByKey() aggregates all values for a given key.
# For key 200, it will compute: (200, 1+1+1+...)
status_counts = status_counts_rdd.reduceByKey(lambda x, y: x + y)

Collect Results and Print

In [38]:
#Collect and Print Results
# Let's see the top 10 most frequent status codes
top_10_status_codes = status_counts.takeOrdered(10, key=lambda x: -x[1])
print("--- Top 10 HTTP Status Code Counts ---")
for status, count in top_10_status_codes:
    print(f"Status Code: {status}, Count: {count}")

--- Top 10 HTTP Status Code Counts ---
Status Code: 200, Count: 1700743
Status Code: 304, Count: 132626
Status Code: 302, Count: 46569
Status Code: 404, Count: 10783
Status Code: 500, Count: 62
Status Code: 403, Count: 54
Status Code: 501, Count: 14


### Log Processing using Functions

Craft pur functiosn to parse the lines, get the staus category and add counts.

In [39]:
# --- Pure Python Functions for RDD Operations ---

def parse_log_line(line):
    """
    Parses a single line from the NASA log file using a regular expression.

    Args:
        line (str): A single line from the log file.

    Returns:
        tuple: A tuple of (status_code, 1) if parsing is successful.
        None: If the line is malformed and does not match the pattern.
    """
    match = re.search(LOG_PATTERN, line)
    if match:
        status_code = int(match.group(8))
        return (status_code, 1)
    else:
        return None

In [40]:
def is_valid_log(record):
    """
    A pure Python function to check if a parsed record is valid (not None).
    This replaces a lambda function in the filter operation.
    """
    return record is not None

In [41]:
def get_status_category(status_code):
    """
    Classifies an HTTP status code into a general category.
    """
    if 200 <= status_code < 300:
        return 'Success'
    elif 300 <= status_code < 400:
        return 'Redirect'
    elif 400 <= status_code < 500:
        return 'Client Error'
    elif 500 <= status_code < 600:
        return 'Server Error'
    else:
        return 'Unknown'

In [42]:
def add(x, y):
    """
    A pure Python function to add two numbers.
    This replaces a lambda function in the reduceByKey operation.
    """
    return x + y

Code rewrite

In [43]:
try:
    # Step A: Parse each line using the map transformation.
    parsed_logs_rdd = log_rdd.map(parse_log_line)

    # Step B: Filter out malformed lines using our custom function.
    valid_logs_rdd = parsed_logs_rdd.filter(is_valid_log)

    # Step C: Classify each status code into a category.
    # The RDD is transformed from (200, 1) to ('Success', 1).
    category_rdd = valid_logs_rdd.map(lambda tpl: (get_status_category(tpl[0]), tpl[1]))

    # Step D: Count the occurrences of each category using our 'add' function.
    category_counts = category_rdd.reduceByKey(add)

    # 4. Collect and Print Results
    results = category_counts.collect()

    print("--- HTTP Status Code Category Counts from NASA Logs ---")
    print("-----------------------------------------------------")
    for category, count in sorted(results, key=lambda x: -x[1]):
        print(f"Category: {category:<15} Count: {count:,}")
    print("-----------------------------------------------------")

except Exception as e:
    print(f"An error occurred: {e}")

--- HTTP Status Code Category Counts from NASA Logs ---
-----------------------------------------------------
Category: Success         Count: 1,700,743
Category: Redirect        Count: 179,195
Category: Client Error    Count: 10,837
Category: Server Error    Count: 76
-----------------------------------------------------


Stop

In [44]:
# spark.stop()

End of Use Case